# Linear regression, workshop 3 — The learning rate, and why scaling exists

The subject asks three questions and leaves you to find out:

> *How does the learning rate and the number of iterations influence the result?*
> *How to tune hyperparameters and choose a good learning rate?*
> *What about the number of iterations? What is the convergence rate?*

Gradient descent has exactly one knob you must set yourself, and it decides everything. There is
no formula for the right value — but there are **four recognisable behaviours**, and once you can
name them from a loss curve you fix a run in one try instead of ten.

1. **Four regimes**, and the exact value where it tips over.
2. **Convergence rate** — what "it converges" actually promises.
3. **Why unscaled features break descent** — the same algorithm, 1000x worse.
4. **Diagnosing a run from its loss curve alone.**

In [ ]:
# ABOUTME: My Linear Regression workshop 3 - learning rate regimes on the subject's f.
# ABOUTME: Convergence rate, feature scaling, and loss-curve diagnosis.

import numpy as np
import matplotlib.pyplot as plt

# The data the project subject generates, verbatim. A seed is added so every run of this notebook
# shows the same numbers; the subject does not fix one, so yours will differ in the last decimals.
np.random.seed(42)
X = 4 * np.random.rand(100, 1)
y = 10 + 2 * X + np.random.randn(100, 1)

def ready(*values):
    """True once the exercises a cell depends on have been filled in."""
    return all(v is not None for v in values)

print("X", X.shape, " y", y.shape)
print("generated with intercept 10 and slope 2, plus standard normal noise")

In [ ]:
TARGET = np.array([2.0, 6.0])      # where the subject's f has its minimum

def f(x):
    """f(x) = 3 + (x - [2, 6])^T (x - [2, 6]). Minimum value 3, reached at [2, 6]."""
    d = np.asarray(x, dtype=float) - TARGET
    return float(3 + d @ d)

def fprime(x):
    """Its gradient: 2(x - [2, 6])."""
    return 2 * (np.asarray(x, dtype=float) - TARGET)

print("f at the minimum:", f(TARGET), "| f at the origin:", f([0.0, 0.0]))

def descend(start, learning_rate, iterations=50, grad=fprime):
    """Plain gradient descent, keeping the whole path."""
    x = np.array(start, dtype=float)
    path = [x.copy()]
    for _ in range(iterations):
        x = x - learning_rate * grad(x)
        path.append(x.copy())
    return np.array(path)

## 1. Four regimes

We know the answer — minimum 3 at [2, 6] — so any disagreement is the algorithm's fault.

### Exercise 1 — run the same descent at six learning rates

In [ ]:
rates = [0.01, 0.1, 0.5, 0.9, 1.0, 1.1]

# TODO: for each rate, run 50 steps from [0, 0] and record the final point and f there
table = None
table

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("six rates tried", lambda: len(table) == 6)
check("0.1 converges", lambda: abs(table.loc[table["learning rate"] == 0.1, "f(theta)"].item() - 3) < 1e-6)
check("1.0 does NOT", lambda: table.loc[table["learning rate"] == 1.0, "f(theta)"].item() > 40)
check("1.1 explodes", lambda: table.loc[table["learning rate"] == 1.1, "f(theta)"].item() > 1e6)

Four distinct behaviours, and each has a name you should recognise instantly:

| learning rate | what happens | the symptom |
|---|---|---|
| **0.01** | right direction, far too slow — still at `[1.272, 3.815]` after 50 steps | loss still falling when you run out of patience |
| **0.1 – 0.9** | converges to `[2, 6]`, f = 3 | loss flattens at the right level |
| **1.0** | **oscillates for ever**, landing exactly back at the start, f = 43 | loss constant and non-zero, step after step |
| **1.1** | diverges to about `[-18 199, -54 597]`, f ≈ 3.3e9 | loss grows, then `inf`, then `nan` |

The tipping point is not folklore. This bowl has second derivative 2 in both directions, so one
step multiplies the distance to the minimum by $(1 - 2\alpha)$:

- $\alpha < 1$: the factor is smaller than 1 in magnitude — the distance shrinks, convergence.
- $\alpha = 1$: the factor is exactly $-1$ — each step lands the same distance on the *other* side,
  for ever. After an even number of steps you are back where you began, which is why the table
  shows `[0, 0]`.
- $\alpha > 1$: the factor exceeds 1 and the distance grows every step.

**Rule of thumb:** when your loss turns into `nan`, suspect the learning rate before your gradient.

### Exercise 2 — find the edge by measuring

In [ ]:
def converges(lr, iterations=200, tolerance=1e-3):
    """True if 200 steps from [0, 0] end within `tolerance` of the known minimum of 3."""
    # TODO: run descend, then check f(end) is finite and close to 3
    return None

# TODO: scan 0.01 to 1.50 in steps of 0.01 and keep the largest value that converges
largest_working_lr = None
print("largest learning rate that still converges:", largest_working_lr)

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("an answer was found", lambda: largest_working_lr is not None)
check("it sits just below 1.0", lambda: 0.90 <= largest_working_lr < 1.0)
check("a little more diverges", lambda: not converges(largest_working_lr + 0.05))

You land on **0.98**, not 1.0 — and the gap answers the subject's question about iterations.

Theory says anything below 1.0 converges *eventually*. Your test also demands arriving **within 200
steps**. At 0.99 each step shrinks the distance by only 2 %, so after 200 steps it is still outside
the tolerance. **Convergence is a promise about the limit; your patience is a budget.**

## 2. Convergence rate

The subject asks what the convergence rate is. You can now answer it precisely: the distance to the
minimum is multiplied by a constant factor every step, so the error falls **geometrically** — a
straight line on a log plot. At $\alpha = 0.1$ the factor is $|1 - 2(0.1)| = 0.8$: ten steps buy
you roughly one decimal digit.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for lr in (0.05, 0.1, 0.3, 0.5):
    path = descend([0.0, 0.0], lr, 60)
    ax.plot([f(p) - 3 for p in path], lw=2, label=f"lr = {lr}")
ax.set_yscale("log"); ax.set_xlabel("iteration"); ax.set_ylabel("f(theta) - 3  (distance to the minimum)")
ax.set_title("Geometric convergence: straight lines on a log scale"); ax.legend()
plt.tight_layout()

Straight lines, different slopes. That is what "linear convergence rate" means — and it is also why
a learning rate that is merely *small* is not safe: it is a slope so shallow you never arrive.

## 3. Why scaling exists

Now the subject's regression, with one change: the feature is recorded in different units — the
same information, numbers 1000x larger. Nothing about the problem changes. Watch the algorithm fall
apart anyway.

In [ ]:
X_new = np.hstack([np.ones((X.shape[0], 1)), X])
X_big = np.hstack([np.ones((X.shape[0], 1)), X * 1000])
m = len(X_new)

def make_cost_grad(design):
    cost = lambda th: float(np.mean((design.dot(th.reshape(-1, 1)) - y) ** 2))
    def grad(th):
        return (2 / m * design.T.dot(design.dot(th.reshape(-1, 1)) - y)).ravel()
    return cost, grad

cost_ok, grad_ok = make_cost_grad(X_new)
cost_big, grad_big = make_cost_grad(X_big)

for label, cost, grad in [("original units", cost_ok, grad_ok), ("x1000 units", cost_big, grad_big)]:
    with np.errstate(over="ignore", invalid="ignore"):
        end = descend([0.0, 0.0], 0.1, 200, grad=grad)[-1]
    print(f"{label:15} lr=0.1 -> theta {np.array2string(end, precision=2)}  cost {cost(end):.4g}")

Identical data, identical algorithm, identical learning rate. One converges; the other overflows.

The reason is in the gradient: its second component is the error **weighted by the feature**.
Multiply the feature by 1000 and that partial derivative grows by 1000, so a step size that suits
the intercept is a thousand times too large for the slope. The round bowl from workshop 2 has
become a canyon.

Two fixes, and you should know both:

1. **Scale the features** — subtract the mean, divide by the standard deviation. Every parameter
   then lives on a comparable scale and one learning rate suits them all.
2. **Use a smaller learning rate** — possible in theory, hopeless in practice: it must be small
   enough for the worst-scaled parameter, which makes every other one crawl.

### Exercise 3 — scale and retry

In [ ]:
def standardise(column):
    """Return (scaled column, mean, std) - the classic z-score."""
    # TODO
    return None, None, None

scaled, mu, sigma = standardise(X[:, 0] * 1000)
if ready(scaled):
    X_scaled = np.hstack([np.ones((len(X), 1)), scaled.reshape(-1, 1)])
    cost_s, grad_s = make_cost_grad(X_scaled)
    end = descend([0.0, 0.0], 0.1, 200, grad=grad_s)[-1]
    print("scaled, lr=0.1 ->", np.round(end, 3), " cost", round(cost_s(end), 4))

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("mean 0", lambda: abs(float(np.mean(scaled))) < 1e-9)
check("std 1", lambda: abs(float(np.std(scaled)) - 1) < 1e-9)
check("mean and std returned for later", lambda: mu is not None and sigma is not None)

The run that overflowed now converges, at the same learning rate, to the same cost.

One caveat worth saying out loud: your parameters are now in **scaled units**. The slope means
"change per standard deviation", not per unit of x. To report in the original units — or to predict
for a new x — you must apply the same transformation with the **same mean and standard deviation**
you stored at training time. That is exactly why scikit-learn scalers separate `fit` from
`transform`.

## 4. Reading a loss curve

In real training you cannot see the cost surface — your model has thousands of parameters, not two.
The loss curve is all you get, and it is enough.

In [ ]:
def curve(lr, iterations=60):
    with np.errstate(over="ignore", invalid="ignore"):
        return np.array([f(p) for p in descend([0.0, 0.0], lr, iterations)])

mystery = {"A": curve(0.005), "B": curve(0.3), "C": curve(1.0), "D": curve(1.05)}

# One shared axis: with a per-panel axis, a dead-flat curve autoscales into a dramatic slope
# and the whole point is lost.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, (label, c) in zip(axes, mystery.items()):
    ax.plot(np.where(np.isfinite(c), c, np.nan), lw=2)
    ax.axhline(3, color="k", ls="--", lw=1)
    ax.set_yscale("log"); ax.set_ylim(1, 1e5)
    ax.set_title(f"run {label}"); ax.set_xlabel("iteration")
axes[0].set_ylabel("f (log scale)")
axes[0].annotate("the minimum, 3", xy=(2, 3.4), fontsize=8)
plt.tight_layout()

### Exercise 4 — name each failure

- `"too small"` — right direction, far too slowly
- `"good"` — drops fast, flattens at the minimum
- `"stuck, bouncing in place"` — a flat line at a *high* value: each step lands the same distance
  on the other side, so nothing improves, ever
- `"diverged"` — climbs, order of magnitude after order of magnitude

Careful with the two flat curves: one is flat because it arrived, the other because it is bouncing.
Only the **value on the axis** tells them apart — which is the whole reason to draw the minimum on
the plot.

In [ ]:
diagnosis = {"A": "", "B": "", "C": "", "D": ""}
diagnosis

In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

_final = {k: (v[np.isfinite(v)][-1] if np.isfinite(v).any() else np.inf) for k, v in mystery.items()}
check("A is still far from the minimum", lambda: _final["A"] > 3.5)
check("B arrived", lambda: abs(_final["B"] - 3) < 1e-3)
check("C never moved", lambda: abs(_final["C"] / mystery["C"][0] - 1) < 1e-9)
check("D blew up", lambda: _final["D"] > 1e6)
check("you labelled all four", lambda: all(diagnosis.values()))

Run C is the one worth staring at: perfectly flat, at **43**, where the minimum is 3. A beginner
reports "the loss stopped changing, it converged". It did not — it is bouncing between two points
on opposite walls, for ever.

**Rule of thumb:** a loss curve on a **log scale, with the achievable minimum drawn on it**,
separates "slow" from "stuck" from "unstable" at a glance. One line of code, hours saved.

## What to take away

1. **Four regimes** — too small, good, oscillating, diverging — recognisable from the loss curve.
2. **The tipping point is set by the curvature.** On this f it is exactly 1.0, and you measured it.
3. **`nan` means the learning rate first**, your gradient second.
4. **Convergence is geometric**: a constant factor per step, a straight line on a log plot. "It
   converges" says nothing about whether it converges *in your lifetime*.
5. **Scaling is not cosmetic.** The same run went from overflow to converged, unchanged otherwise.
6. **Scaled parameters are in scaled units.** Keep the mean and std; you need them at prediction time.
7. **Plot the loss on a log scale with the target drawn in**, every run.

### Take it home
Reproduce the six-rate table and the four loss curves in your own notebook — they are the direct
answer to the three questions the subject asks, and a reviewer can read them in ten seconds.